# List-aware · CatBoost × XGBoost model showdown

**Goal:** compare our historically strongest list-aware approaches on **exactly the same chronological properties**, including hybrids of CatBoost Huber with XGBoost's best spline/geographic feature representations and fixed-weight multi-model blends. No neural networks.

The previously recorded **4.06% MdAPE** CatBoost Huber score used a different, full-scale historical evaluation. **Do not compare that number directly to a capped pilot.** This notebook fits a matched CatBoost Huber control in the same pilot.

The experiment delegates training to `listaware_shotgun.py` and uses the parameter configurations in `models/listaware_shotgun_specs.py`. Completed individual predictions are cached per member and month; rerunning with unchanged settings resumes without refitting them.

Full guide: [listaware_shotgun.md](docs/listaware_shotgun.md).

## 1 · Repository and execution environment

Use a Python kernel with the project's dependencies. Local Windows/VS Code works from the project root. Google Colab can clone the public code; it needs the normal competition listings dataset (cached/downloaded by `data.py`).

In [1]:
from pathlib import Path
import os
import sys
import subprocess

PROJECT_ROOT = None  # Optional manual repository checkout path.
DATA_DIR_OVERRIDE = None  # None preserves current environment; important for Parquet parity.
CLONE_IN_COLAB_IF_MISSING = True
INSTALL_REQUIREMENTS_IN_COLAB = True

def locate_project():
    if PROJECT_ROOT is not None:
        path = Path(PROJECT_ROOT).expanduser().resolve()
        if not (path / "listaware_shotgun.py").is_file():
            raise FileNotFoundError(f"Not a project checkout: {path}")
        return path
    current = Path.cwd().resolve()
    for parent in (current, *current.parents):
        for candidate in (parent, parent / "SSU-Data-Dash-hdnl-comp"):
            if (candidate / "listaware_shotgun.py").is_file():
                return candidate
    if "google.colab" in sys.modules and CLONE_IN_COLAB_IF_MISSING:
        target = Path("/content/SSU-Data-Dash-hdnl-comp")
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/HiddenL123/SSU-Data-Dash-hdnl-comp.git",
             str(target)],
            check=True,
        )
        return target
    raise FileNotFoundError("Open from the repository or set PROJECT_ROOT.")

ROOT = locate_project()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if DATA_DIR_OVERRIDE:
    os.environ["DATA_DIR"] = str(Path(DATA_DIR_OVERRIDE).expanduser().resolve())
if "google.colab" in sys.modules and INSTALL_REQUIREMENTS_IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(ROOT / "requirements.txt")], check=True)
print("Repository:", ROOT)
print("Python:", sys.version.split()[0])
print("DATA_DIR:", os.environ.get("DATA_DIR", "(project data/ or Hugging Face)"))

Repository: C:\Users\deant\Desktop\Coding\Data Dash Case Comp
Python: 3.13.11
DATA_DIR: (project data/ or Hugging Face)


## 2 · Inspect the recorded candidates and fixed blends

The key **cross-family transfer** is CatBoost Huber trained on the feature set developed for the strong XGBoost off-market model—not simply averaging the original models' predictions. Other arms train tuned XGBoost or LightGBM on their own backends.

Additional blended arms average **predicted sale prices in log space** using weights fixed before seeing these fold results.

In [2]:
import pandas as pd
import numpy as np
from IPython.display import display
from models.listaware_shotgun_specs import CANDIDATES, DEFAULT_CANDIDATES, BLENDS

catalog = pd.DataFrame([{
    "candidate": key, "backend": c.backend, "target": c.target,
    "loss": c.loss, "feature_set": c.feature_name,
    "tuned": c.tuned, "in_default": key in DEFAULT_CANDIDATES,
} for key, c in CANDIDATES.items()])
display(catalog)
blend_catalog = pd.DataFrame([
    {"blend": name, "members": ", ".join(f"{m}: {w:.0%}" for m, w in weights.items())}
    for name, weights in BLENDS.items()
])
display(blend_catalog)

,candidate,backend,target,loss,feature_set,tuned,in_default
0,cat_huber_baseline,catboost,ratio,Huber:delta=0.05,Xpt_spatiotemporal,False,True
1,cat_huber_binary,catboost,ratio,Huber:delta=0.05,Xpt_spatiotemporal__selected_binary,False,True
2,cat_huber_spline_geo,catboost,ratio,Huber:delta=0.05,Xpt_spatiotemporal__selected_binary__st_spacet...,False,True
3,cat_huber_adjusted,catboost,ratio,Huber:delta=0.05,Xpt_spatiotemporal__selected_binary__st_spacet...,False,True
4,cat_huber_advanced,catboost,ratio,Huber:delta=0.05,Xpt_spatiotemporal__selected_binary__st_spacet...,False,True
5,cat_huber_hedonic,catboost,ratio,Huber:delta=0.05,Xpt_spatiotemporal__selected_binary__st_spacet...,False,False
6,cat_huber_adaptive,catboost,ratio,Huber:delta=0.05,Xpt_spatiotemporal__selected_binary__st_spacet...,False,False
7,cat_huber_time_adjusted,catboost,ratio,Huber:delta=0.05,Xpt_spatiotemporal__selected_binary__st_spacet...,False,False
8,cat_mae,catboost,ratio,MAE,Xpt_spatiotemporal,False,True
9,cat_rmse,catboost,ratio,RMSE,Xpt_spatiotemporal,False,False


,blend,members
0,blend_cat_baseline_xgb,"cat_huber_baseline: 70%, xgb_tuned_binary: 30%"
1,blend_cat_xgb_spatial,"cat_huber_spline_geo: 70%, xgb_spline_geo: 30%"
2,blend_cat_xgb_adjusted,"cat_huber_adjusted: 65%, xgb_adjusted: 35%"
3,blend_cat_xgb_lgb,"cat_huber_spline_geo: 50%, xgb_spline_geo: 35%..."
4,blend_cat_baseline_spatial_xgb,"cat_huber_baseline: 40%, cat_huber_spline_geo:..."


## 3 · Select the experiment

Set `SCREEN = "broad"` for the default multi-family shotgun; set `"cross_family"` for a smaller CatBoost/XGBoost-specific test, or `"objectives"` to isolate the target and loss.

The default pilot uses **August–October 2025**, 10,000 training properties per month, 350 validation homes per month, and at most 300 boosting rounds per model. It uses exactly matched evaluation rows and the same chronological training restrictions for every candidate.

If you change models, months or budgets after an earlier run, change `OUTPUT_DIR` as well. Model caches are keyed by a strict experiment manifest.

In [3]:
from pathlib import Path
from listaware_shotgun import run_listaware_shotgun

SCREEN = "broad"  # "broad", "cross_family", "objectives"
SCREENS = {
    "broad": tuple(DEFAULT_CANDIDATES),
    "cross_family": (
        "cat_huber_baseline", "cat_huber_spline_geo", "xgb_spline_geo",
        "lgb_spline_geo",
    ),
    "objectives": (
        "cat_huber_baseline", "cat_huber_direct", "cat_mae",
        "cat_rmse", "cat_tuned_rmse",
    ),
}
CANDIDATES_TO_TEST = SCREENS[SCREEN]
MONTHS = ("2025-08", "2025-09", "2025-10")
MODE = "expanding"  # Optionally "fixed_origin" with a DIFFERENT output folder.
OUTPUT_DIR = Path(f"outputs/listaware_notebook_{SCREEN}_{MODE}_pilot")

RUN_TESTS = False
RUN_PILOT = True
WITH_BLENDS = True
TRAIN_CAP = 10_000
VALIDATION_CAP = 350
TREE_CAP = 300
THREADS = 4
BOOTSTRAP_REPEATS = 300
SEED = 42

assert "cat_huber_baseline" in CANDIDATES_TO_TEST
assert set(CANDIDATES_TO_TEST).issubset(CANDIDATES)
print("Candidate count:", len(CANDIDATES_TO_TEST))
print("Active fixed blends:", [name for name, w in BLENDS.items()
                              if WITH_BLENDS and set(w).issubset(CANDIDATES_TO_TEST)])
print("Output:", OUTPUT_DIR)
display(catalog.loc[catalog.candidate.isin(CANDIDATES_TO_TEST)])

Candidate count: 11
Active fixed blends: ['blend_cat_baseline_xgb', 'blend_cat_xgb_spatial', 'blend_cat_xgb_adjusted', 'blend_cat_xgb_lgb', 'blend_cat_baseline_spatial_xgb']
Output: outputs\listaware_notebook_broad_expanding_pilot


,candidate,backend,target,loss,feature_set,tuned,in_default
0,cat_huber_baseline,catboost,ratio,Huber:delta=0.05,Xpt_spatiotemporal,False,True
1,cat_huber_binary,catboost,ratio,Huber:delta=0.05,Xpt_spatiotemporal__selected_binary,False,True
2,cat_huber_spline_geo,catboost,ratio,Huber:delta=0.05,Xpt_spatiotemporal__selected_binary__st_spacet...,False,True
3,cat_huber_adjusted,catboost,ratio,Huber:delta=0.05,Xpt_spatiotemporal__selected_binary__st_spacet...,False,True
4,cat_huber_advanced,catboost,ratio,Huber:delta=0.05,Xpt_spatiotemporal__selected_binary__st_spacet...,False,True
8,cat_mae,catboost,ratio,MAE,Xpt_spatiotemporal,False,True
12,xgb_tuned_binary,xgboost,ratio,RMSE,Xpt_spatiotemporal__selected_binary,True,True
13,xgb_spline_geo,xgboost,ratio,RMSE,Xpt_spatiotemporal__selected_binary__st_spacet...,True,True
14,xgb_adjusted,xgboost,ratio,RMSE,Xpt_spatiotemporal__selected_binary__st_spacet...,True,True
16,lgb_tuned_binary,lightgbm,ratio,RMSE,Xpt_spatiotemporal__selected_binary,True,True


## 4 · Input and regression-test preflight

The competition's historic `listings` file supplies all labels. Validation homes are passed to the model **without `sold_price` and `date_sold`**, but their **list prices remain visible**. Training includes only sales completed before the prediction month's cutoff.

The tests are optional in the notebook because the full experiment already validates schema and cohort alignment; run them at least once locally before relying on results.

In [4]:
import data

if RUN_TESTS:
    subprocess.run([
        sys.executable, "-m", "pytest",
        "tests/test_listaware_shotgun.py",
        "tests/test_realworld_avm_shotgun.py", "-q",
    ], check=True, cwd=ROOT)

print("Listings source:", data.find_file("listings"))
print("Train cutoff: earlier completed sales only")
print("Screen:", SCREEN)
print("Evaluation months:", MONTHS)
print("Target reconstruction: exp(predicted log sale/list ratio) * list_price")

Listings source: C:\Users\deant\Desktop\Coding\Data Dash Case Comp\data\listings.csv
Train cutoff: earlier completed sales only
Screen: broad
Evaluation months: ('2025-08', '2025-09', '2025-10')
Target reconstruction: exp(predicted log sale/list ratio) * list_price


## 5 · Run or resume the pilot

The first execution fits each candidate and saves a separate CSV **immediately after each completed member/month**. Rerun with identical settings to resume. `RUN_PILOT = False` loads completed CSVs without retraining.

In [5]:
if RUN_PILOT:
    pooled, monthly, by_type, predictions = run_listaware_shotgun(
        months=MONTHS,
        candidates=CANDIDATES_TO_TEST,
        full=False,
        mode=MODE,
        train_cap=TRAIN_CAP,
        validation_cap=VALIDATION_CAP,
        trees=TREE_CAP,
        threads=THREADS,
        seed=SEED,
        with_blends=WITH_BLENDS,
        bootstrap_repeats=BOOTSTRAP_REPEATS,
        output_dir=OUTPUT_DIR,
    )
else:
    pooled = pd.read_csv(OUTPUT_DIR / "pooled_scores.csv")
    monthly = pd.read_csv(OUTPUT_DIR / "monthly_scores.csv")
    by_type = pd.read_csv(OUTPUT_DIR / "house_type_scores.csv")
    predictions = pd.read_csv(OUTPUT_DIR / "predictions.csv",
                              dtype={"ml_num": str})
print("Prediction rows:", len(predictions))

2025-08 cat_huber_baseline: fitting on 10,000 past sales / 350 unseen homes
  finished: MdAPE=4.6394% in 51.3s
2025-08 cat_huber_binary: fitting on 10,000 past sales / 350 unseen homes
  finished: MdAPE=4.5552% in 102.0s
2025-08 cat_huber_spline_geo: fitting on 10,000 past sales / 350 unseen homes
  finished: MdAPE=4.7172% in 95.5s
2025-08 cat_huber_adjusted: fitting on 10,000 past sales / 350 unseen homes
  finished: MdAPE=4.8028% in 114.8s
2025-08 cat_huber_advanced: fitting on 10,000 past sales / 350 unseen homes
  finished: MdAPE=4.6558% in 267.0s
2025-08 cat_mae: fitting on 10,000 past sales / 350 unseen homes
  finished: MdAPE=4.4693% in 256.4s
2025-08 xgb_tuned_binary: fitting on 10,000 past sales / 350 unseen homes
  finished: MdAPE=4.9492% in 130.4s
2025-08 xgb_spline_geo: fitting on 10,000 past sales / 350 unseen homes
  finished: MdAPE=4.7962% in 155.5s
2025-08 xgb_adjusted: fitting on 10,000 past sales / 350 unseen homes
  finished: MdAPE=4.9532% in 176.3s
2025-08 lgb_tuned

## 6 · Leaderboard: models and hybrids

`delta_mdape_pp` is relative to **CatBoost Huber trained and scored on the identical pilot cohort**. Negative means better. MdAPE is the primary competition metric, with MAPE and tail behaviour useful for secondary evaluation.

Use `delta_mdape_ci_low` / `delta_mdape_ci_high` as an exploratory paired bootstrap diagnostic, not a guarantee of an untouched test win.

In [6]:
columns = [
    "arm", "variant", "homes", "mdape", "mape", "p90_ape",
    "delta_mdape_pp", "delta_mape_pp",
    "homes_improved_fraction",
    "delta_mdape_ci_low", "delta_mdape_ci_high",
]
display(pooled[[c for c in columns if c in pooled]].round(4).reset_index(drop=True))
reference = pooled.loc[
    (pooled.arm == "model") & (pooled.variant == "cat_huber_baseline")
]
assert len(reference) == 1
print("Matched CatBoost Huber baseline MdAPE:",
      f"{reference.mdape.iloc[0]:.4f}%")
best_row = pooled.sort_values(["mdape", "mape"]).iloc[0]
print(f"Best candidate: {best_row.variant} ({best_row.arm}) — "
      f"MdAPE {best_row.mdape:.4f}%")

,arm,variant,homes,mdape,mape,p90_ape,delta_mdape_pp,delta_mape_pp,homes_improved_fraction,delta_mdape_ci_low,delta_mdape_ci_high
0,model,cat_huber_spline_geo,1050,4.2899,5.5044,11.6205,-0.1251,-0.0490,0.5171,-0.2408,0.0689
1,model,cat_huber_advanced,1050,4.3061,5.4989,11.5886,-0.1089,-0.0545,0.5219,-0.2105,0.1196
2,model,cat_mae,1050,4.3586,5.5205,11.5616,-0.0564,-0.0329,0.5200,-0.1744,0.1134
3,model,cat_huber_baseline,1050,4.4150,5.5535,11.5541,0.0000,0.0000,0.0000,0.0000,0.0000
4,blend,blend_cat_baseline_xgb,1050,4.4341,5.4909,11.2822,0.0191,-0.0625,0.5295,-0.1133,0.2935
5,model,cat_huber_binary,1050,4.4342,5.5052,11.2782,0.0192,-0.0483,0.5600,-0.1274,0.1547
6,model,cat_huber_adjusted,1050,4.4388,5.5376,11.5011,0.0238,-0.0159,0.5086,-0.0744,0.2547
7,blend,blend_cat_xgb_spatial,1050,4.4585,5.5006,11.3401,0.0435,-0.0529,0.5295,-0.1220,0.2630
8,model,lgb_tuned_binary,1050,4.4606,5.5837,11.4318,0.0456,0.0303,0.4981,-0.1943,0.3627
9,blend,blend_cat_baseline_spatial_xgb,1050,4.4678,5.5081,11.4360,0.0527,-0.0453,0.5410,-0.1030,0.2313


Matched CatBoost Huber baseline MdAPE: 4.4150%
Best candidate: cat_huber_spline_geo (model) — MdAPE 4.2899%


In [7]:
import matplotlib.pyplot as plt

ordered = pooled.sort_values("delta_mdape_pp")
fig, ax = plt.subplots(figsize=(9, max(4, .40 * len(ordered))))
labels = ordered["variant"].astype(str) + " [" + ordered["arm"].astype(str) + "]"
ax.barh(labels, ordered.delta_mdape_pp)
ax.axvline(0, color="gray", linewidth=1)
ax.invert_yaxis()
ax.set_title("List-aware: paired MdAPE difference vs CatBoost Huber")
ax.set_xlabel("Change in MdAPE (percentage points; negative = better)")
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib.backends.registry'

## 7 · Monthly and property-type stability

The pooled metric can mask a big win in just one month or one class of homes. For shortlist selection, prefer models whose paired improvement repeats across months and doesn't create a severe tail-risk trade-off.

In [9]:
display(monthly.pivot_table(
    index=["arm", "variant"], columns="month", values="delta_mdape_pp"
).round(3))
if len(by_type):
    print("By house type (positive delta = worse than CatBoost Huber):")
    display(by_type.pivot_table(
        index=["arm", "variant"], columns="house_type_name",
        values="delta_mdape_pp",
    ).round(3))

month                                 2025-08  2025-09  2025-10
arm   variant                                                  
blend blend_cat_baseline_spatial_xgb    0.095    0.022   -0.193
      blend_cat_baseline_xgb            0.247    0.021   -0.183
      blend_cat_xgb_adjusted            0.242    0.236    0.045
      blend_cat_xgb_lgb                 0.113    0.219   -0.104
      blend_cat_xgb_spatial             0.134    0.080   -0.122
model cat_huber_adjusted                0.163    0.003   -0.200
      cat_huber_advanced                0.016   -0.176   -0.217
      cat_huber_baseline                0.000    0.000    0.000
      cat_huber_binary                 -0.084    0.028   -0.078
      cat_huber_spline_geo              0.078    0.004   -0.431
      cat_mae                          -0.170    0.030   -0.174
      lgb_spline_geo                    0.037    0.195   -0.037
      lgb_tuned_binary                  0.032   -0.107   -0.010
      xgb_adjusted                      0.314    0.669   -0.135
      xgb_spline_geo                    0.157    0.447    0.022
      xgb_tuned_binary                  0.310    0.130    0.059

By house type (positive delta = worse than CatBoost Huber):


house_type_name                       Condo Apt  Condo Townhouse  Detached  \
arm   variant                                                                
blend blend_cat_baseline_spatial_xgb     -0.081            0.056    -0.090   
      blend_cat_baseline_xgb             -0.054            0.027     0.116   
      blend_cat_xgb_adjusted              0.115            0.181    -0.022   
      blend_cat_xgb_lgb                   0.008            0.234     0.007   
      blend_cat_xgb_spatial              -0.022            0.093    -0.101   
model cat_huber_adjusted                  0.104           -0.332    -0.060   
      cat_huber_advanced                 -0.032            0.040    -0.067   
      cat_huber_baseline                  0.000            0.000     0.000   
      cat_huber_binary                   -0.195           -0.033    -0.154   
      cat_huber_spline_geo               -0.050           -0.169    -0.175   
      cat_mae                             0.038           -0.346     0.042   
      lgb_spline_geo                      0.112            0.020     0.361   
      lgb_tuned_binary                   -0.088            0.085     0.411   
      xgb_adjusted                        0.212            0.339     0.453   
      xgb_spline_geo                     -0.012            0.149     0.472   
      xgb_tuned_binary                    0.198            0.143     0.111   

house_type_name                       Freehold Townhouse  Semi-Detached  
arm   variant                                                            
blend blend_cat_baseline_spatial_xgb               0.083         -0.062  
      blend_cat_baseline_xgb                      -0.667          0.478  
      blend_cat_xgb_adjusted                      -0.150         -0.128  
      blend_cat_xgb_lgb                           -0.360         -0.125  
      blend_cat_xgb_spatial                        0.062         -0.371  
model cat_huber_adjusted                           0.151         -0.364  
      cat_huber_advanced                           0.406         -0.406  
      cat_huber_baseline                           0.000          0.000  
      cat_huber_binary                            -0.542         -0.030  
      cat_huber_spline_geo                         0.173          0.043  
      cat_mae                                      0.255          0.175  
      lgb_spline_geo                              -1.915         -0.144  
      lgb_tuned_binary                            -0.341         -0.353  
      xgb_adjusted                                -1.866          0.036  
      xgb_spline_geo                              -1.191          0.118  
      xgb_tuned_binary                            -1.277          0.762

## 8 · Do CatBoost and XGBoost actually complement one another?

Compare blended rows with the best individual members on the same homes. A blended model may have **lower MAPE/P90 but slightly higher MdAPE**; this matters because the competition emphasizes the median.

The recorded blends have *precommitted weights*. Do not optimize new weights on these same months and then describe that score as untouched validation.

In [10]:
model_scores = pooled.loc[pooled.arm == "model"].sort_values(["mdape", "mape"])
blend_scores = pooled.loc[pooled.arm == "blend"].sort_values(["mdape", "mape"])
print("Best single models:")
display(model_scores[["variant", "mdape", "mape", "p90_ape", "delta_mdape_pp"]].head(10).round(4))
if len(blend_scores):
    print("Predeclared cross-family blends:")
    display(blend_scores[["variant", "mdape", "mape", "p90_ape", "delta_mdape_pp"]].round(4))
else:
    print("No complete predeclared blend was available with these candidates.")

# On the identical cohort, which homes does the top challenger improve?
control = predictions.loc[
    (predictions.arm == "model") & (predictions.variant == "cat_huber_baseline")
].set_index(["month", "ml_num"]).sort_index()
winner = predictions.loc[
    (predictions.arm == best_row.arm)
    & (predictions.variant == best_row.variant)
].set_index(["month", "ml_num"]).sort_index()
paired_error = pd.DataFrame({
    "baseline_ape": control["ape"],
    "candidate_ape": winner.loc[control.index, "ape"],
})
paired_error["improvement_pp"] = paired_error.baseline_ape - paired_error.candidate_ape
display(paired_error.describe(percentiles=[.1, .5, .9]).round(3))

Best single models:


,variant,mdape,mape,p90_ape,delta_mdape_pp
2,cat_huber_spline_geo,4.2899,5.5044,11.6205,-0.1251
4,cat_huber_advanced,4.3061,5.4989,11.5886,-0.1089
5,cat_mae,4.3586,5.5205,11.5616,-0.0564
0,cat_huber_baseline,4.4150,5.5535,11.5541,0.0000
1,cat_huber_binary,4.4342,5.5052,11.2782,0.0192
3,cat_huber_adjusted,4.4388,5.5376,11.5011,0.0238
9,lgb_tuned_binary,4.4606,5.5837,11.4318,0.0456
10,lgb_spline_geo,4.5516,5.5923,11.2483,0.1366
7,xgb_spline_geo,4.6386,5.7420,11.8685,0.2236
6,xgb_tuned_binary,4.6896,5.6025,11.0136,0.2746


Predeclared cross-family blends:


,variant,mdape,mape,p90_ape,delta_mdape_pp
11,blend_cat_baseline_xgb,4.4341,5.4909,11.2822,0.0191
12,blend_cat_xgb_spatial,4.4585,5.5006,11.3401,0.0435
15,blend_cat_baseline_spatial_xgb,4.4678,5.5081,11.4360,0.0527
13,blend_cat_xgb_adjusted,4.6031,5.5620,11.4509,0.1881
14,blend_cat_xgb_lgb,4.6108,5.5093,11.4341,0.1958


,baseline_ape,candidate_ape,improvement_pp
count,1050.000,1050.000,1050.000
mean,5.553,5.504,0.049
std,4.765,4.806,0.740
min,0.002,0.006,-2.502
10%,0.812,0.788,-0.868
50%,4.415,4.290,0.032
90%,11.554,11.620,0.942
max,37.123,37.638,2.821


## 9 · Full-strength shortlist (explicitly opt in)

After the pilot, choose two to four promising candidates and keep CatBoost Huber as the reference. Full mode uses all eligible historic sales and each model's recorded full boosting-round count (CatBoost 900, tuned XGBoost 700, tuned LightGBM 1,800). This requires more compute.

Run `--mode fixed_origin` with a **separate output folder** to approximate a model trained at one historical cutoff and applied to multiple months without refreshing.

The original 4.06% CatBoost result can only be fairly compared after evaluating the same held-out homes with the same chronology and fitting budget. Even then, December–May has been inspected during earlier development.

In [11]:
RUN_FULL = True
FULL_CANDIDATES = (
    "cat_huber_baseline",
    "cat_huber_spline_geo",
    "cat_huber_advanced",
    "cat_mae",
    "cat_huber_binary",
    "xgb_tuned_binary",
    "xgb_spline_geo",
    "lgb_spline_geo",
)
FULL_MODE = "expanding"
FULL_OUTPUT_DIR = Path(f"outputs/listaware_notebook_full_{FULL_MODE}")

if RUN_FULL:
    full_pooled, full_monthly, full_by_type, full_predictions = run_listaware_shotgun(
        months=MONTHS, candidates=FULL_CANDIDATES,
        full=True, mode=FULL_MODE,
        threads=THREADS, with_blends=WITH_BLENDS,
        bootstrap_repeats=BOOTSTRAP_REPEATS, seed=SEED,
        output_dir=FULL_OUTPUT_DIR,
    )
    display(full_pooled.round(4))
else:
    print("Full-strength training disabled; set RUN_FULL=True to opt in.")

2025-08 cat_huber_baseline: fitting on 132,113 past sales / 1,666 unseen homes
  finished: MdAPE=4.1226% in 703.3s
2025-08 cat_huber_spline_geo: fitting on 132,113 past sales / 1,666 unseen homes
  finished: MdAPE=4.0824% in 546.1s
2025-08 cat_huber_advanced: fitting on 132,113 past sales / 1,666 unseen homes
  finished: MdAPE=4.1480% in 747.0s
2025-08 cat_mae: fitting on 132,113 past sales / 1,666 unseen homes
  finished: MdAPE=4.0694% in 300.3s
2025-08 cat_huber_binary: fitting on 132,113 past sales / 1,666 unseen homes
  finished: MdAPE=4.0422% in 542.9s
2025-08 xgb_tuned_binary: fitting on 132,113 past sales / 1,666 unseen homes
  finished: MdAPE=4.2270% in 220.6s
2025-08 xgb_spline_geo: fitting on 132,113 past sales / 1,666 unseen homes
  finished: MdAPE=4.1371% in 291.0s
2025-08 lgb_spline_geo: fitting on 132,113 past sales / 1,666 unseen homes
  finished: MdAPE=4.1009% in 241.8s
2025-09 cat_huber_baseline: fitting on 133,779 past sales / 2,035 unseen homes
  finished: MdAPE=3.93

,homes,mape,mdape,median_signed,arm,variant,delta_mdape_pp,delta_mape_pp,p90_ape,homes_improved_fraction,delta_mdape_ci_low,delta_mdape_ci_high,delta_mape_ci_low,delta_mape_ci_high
10,5951,4.8227,3.8970,0.6026,blend,blend_cat_xgb_lgb,-0.0559,-0.0986,9.9620,0.5347,-0.1537,0.0155,-0.1345,-0.0595
11,5951,4.8439,3.9156,0.5290,blend,blend_cat_baseline_spatial_xgb,-0.0373,-0.0774,10.0251,0.5431,-0.1143,0.0115,-0.0984,-0.0564
9,5951,4.8425,3.9203,0.5938,blend,blend_cat_xgb_spatial,-0.0326,-0.0788,9.9564,0.5334,-0.0998,0.0293,-0.1061,-0.0520
3,5951,4.9173,3.9214,0.3233,model,cat_mae,-0.0315,-0.0040,10.2264,0.5039,-0.0878,0.0152,-0.0192,0.0110
4,5951,4.9068,3.9347,0.4933,model,cat_huber_binary,-0.0182,-0.0145,10.2442,0.5107,-0.0815,0.0250,-0.0283,-0.0044
2,5951,4.9185,3.9382,0.5437,model,cat_huber_advanced,-0.0148,-0.0028,10.2613,0.5039,-0.0906,0.0389,-0.0214,0.0126
8,5951,4.8539,3.9492,0.5293,blend,blend_cat_baseline_xgb,-0.0037,-0.0675,10.0794,0.5448,-0.0762,0.0572,-0.0926,-0.0451
0,5951,4.9213,3.9529,0.4555,model,cat_huber_baseline,0.0000,0.0000,10.3088,0.0000,0.0000,0.0000,0.0000,0.0000
1,5951,4.9257,3.9706,0.5104,model,cat_huber_spline_geo,0.0177,0.0044,10.2628,0.5045,-0.0416,0.0650,-0.0111,0.0179
6,5951,4.9646,4.0073,0.7781,model,xgb_spline_geo,0.0544,0.0433,10.3727,0.4989,-0.0591,0.1833,-0.0190,0.1191


## 10 · Optional fixed-origin robustness check

The expanding study refits before each development month. To check a harder forecast horizon, you can freeze the training cutoff before August 2025 and predict August–October without updating model training between months.

This creates a **separate experiment**. Since the existing evaluation months have been repeatedly inspected, do not treat either study as an untouched holdout.

In [13]:
RUN_FIXED_ORIGIN = True
FIXED_ORIGIN_OUTPUT = Path("outputs/listaware_notebook_fixed_origin_pilot")

if RUN_FIXED_ORIGIN:
    fixed_pooled, fixed_monthly, fixed_by_type, fixed_predictions = (
        run_listaware_shotgun(
            months=MONTHS,
            candidates=CANDIDATES_TO_TEST,
            full=False,
            mode="fixed_origin",
            train_cap=TRAIN_CAP, validation_cap=VALIDATION_CAP,
            trees=TREE_CAP, threads=THREADS,
            seed=SEED, with_blends=WITH_BLENDS,
            bootstrap_repeats=BOOTSTRAP_REPEATS,
            output_dir=FIXED_ORIGIN_OUTPUT,
        )
    )
    display(fixed_pooled.round(4))
else:
    print("Fixed-origin cross-check disabled.")

2025-08 cat_huber_baseline: cached prediction
2025-08 cat_huber_binary: cached prediction
2025-08 cat_huber_spline_geo: cached prediction
2025-08 cat_huber_adjusted: cached prediction
2025-08 cat_huber_advanced: cached prediction
2025-08 cat_mae: cached prediction
2025-08 xgb_tuned_binary: cached prediction
2025-08 xgb_spline_geo: cached prediction
2025-08 xgb_adjusted: cached prediction
2025-08 lgb_tuned_binary: cached prediction
2025-08 lgb_spline_geo: cached prediction
2025-09 cat_huber_baseline: cached prediction
2025-09 cat_huber_binary: cached prediction
2025-09 cat_huber_spline_geo: cached prediction
2025-09 cat_huber_adjusted: cached prediction
2025-09 cat_huber_advanced: cached prediction
2025-09 cat_mae: cached prediction
2025-09 xgb_tuned_binary: cached prediction
2025-09 xgb_spline_geo: cached prediction
2025-09 xgb_adjusted: cached prediction
2025-09 lgb_tuned_binary: cached prediction
2025-09 lgb_spline_geo: cached prediction
2025-10 cat_huber_baseline: cached prediction


,homes,mape,mdape,median_signed,arm,variant,delta_mdape_pp,delta_mape_pp,p90_ape,homes_improved_fraction,delta_mdape_ci_low,delta_mdape_ci_high,delta_mape_ci_low,delta_mape_ci_high
0,1050,5.6781,4.4974,1.1893,model,cat_huber_baseline,0.0000,0.0000,11.5997,0.0000,0.0000,0.0000,0.0000,0.0000
2,1050,5.6190,4.5372,0.8929,model,cat_huber_spline_geo,0.0397,-0.0591,11.6749,0.5229,-0.2025,0.1585,-0.1047,-0.0105
5,1050,5.6499,4.5586,1.1669,model,cat_mae,0.0611,-0.0282,11.4116,0.5133,-0.1582,0.1993,-0.0695,0.0148
4,1050,5.6239,4.5596,1.0394,model,cat_huber_advanced,0.0622,-0.0541,11.6258,0.5190,-0.1504,0.2205,-0.1013,-0.0106
1,1050,5.6846,4.5887,1.2663,model,cat_huber_binary,0.0913,0.0065,11.4820,0.5048,-0.1345,0.1957,-0.0303,0.0386
15,1050,5.6266,4.6060,1.3634,blend,blend_cat_baseline_spatial_xgb,0.1086,-0.0514,11.4897,0.5457,-0.1314,0.2734,-0.0944,-0.0061
3,1050,5.6278,4.6349,1.0036,model,cat_huber_adjusted,0.1375,-0.0503,11.5903,0.5314,-0.0904,0.2697,-0.0943,-0.0053
11,1050,5.6517,4.6422,1.3474,blend,blend_cat_baseline_xgb,0.1448,-0.0264,11.5253,0.5229,-0.1013,0.2990,-0.0809,0.0259
12,1050,5.6159,4.6735,1.2554,blend,blend_cat_xgb_spatial,0.1761,-0.0622,11.4276,0.5210,-0.1305,0.3260,-0.1211,0.0018
14,1050,5.6272,4.7328,1.4191,blend,blend_cat_xgb_lgb,0.2354,-0.0509,11.3324,0.5343,-0.0959,0.4277,-0.1283,0.0357


## 11 · Promotion checklist

Choose a shortlist by **paired MdAPE**, with MAPE/P90 and per-month consistency as tie-breakers. A successful CatBoost–XGBoost blend can become a candidate for a stronger, fully frozen list-aware ensemble.

**Not included yet:** feeding the full-history off-market model's *in-sample* predictions into list-aware training. That would leak sold prices. A separate two-stage experiment would require **out-of-time off-market predictions for every stage-two training observation** before creating the list-price-versus-market-value feature.

After the two notebook screens finish, send their `pooled_scores.csv` and `pooled_vs_ensemble.csv` results for a controlled winner analysis. The scripts do not overwrite your existing saved models or competition submissions.